In [ ]:
import re
from pathlib import Path

import pandas as pd

In [ ]:
# Prefer an absolute path to the file
base_dir = Path("/home/yisusparker/01_Dev/British_Council/CodingHubs")
file_path = base_dir / "notebooks" / "CHM" / "input" / "Instrumento+EC+CHM+2026_25+de+agosto+de+2026_10.40.xlsx"

if not file_path.exists():
    raise FileNotFoundError(f"File not found: {file_path}")

# Export estilo Qualtrics: fila 0 = códigos de variable, fila 1 = texto de la
# pregunta (header real), fila 2 = metadatos ImportId (se descarta).
codes = pd.read_excel(file_path, header=None, nrows=1).iloc[0].astype(str).tolist()
df_instrumentos = pd.read_excel(file_path, skiprows=[0, 2])

print(df_instrumentos.shape)
df_instrumentos.head()

## Constantes y funciones auxiliares

Portadas de `01_limpieza_datos_EC_2026.py`. El archivo CHM comparte la misma
estructura de bloques (5 anclas de "momento", instantáneas `I1.`...`I18.`, y
los mismos 11 códigos de roster `Q4xx_1`/`Q2xx_1`), así que esta lógica no
necesita cambios.

In [ ]:
# Primera columna de cada bloque de momento; sirve como ancla para detectar los 5 bloques.
ANCHOR = (
    "Transferencia de la experticia - Respuestas - Los/las docentes relatan "
    "anécdotas relacionadas con su enseñanza de pensamiento computacional."
)

# Listado de asistentes: código de Qualtrics -> rol en el roster.
# En el instrumento CHM este roster se llama "CHM1..CHM5" (Coding Hub Master),
# no "PE1..PE5" como en EC — son los mismos 5 códigos, solo cambió el nombre del rol.
ROSTER_CODES = {
    "Q411_1": "CHM1",
    "Q445_1": "CHM2",
    "Q412_1": "CHM3",
    "Q413_1": "CHM4",
    "Q417_1": "CHM5",
    "Q422_1": "DA1",
    "Q437_1": "DA2",
    "Q438_1": "DA3",
    "Q439_1": "DA4",
    "Q440_1": "DA5",
    "Q265_1": "DA6",
}

# Preguntas a nivel de encuentro (condiciones del espacio, fortalezas y mejoras).
CF_STEMS = (
    "condiciones estaban presentes en el espacio",
    "fortalezas considera que se evidenciaron",
    "aspectos considera que podrían mejorarse",
)


def quitar_emoji(s):
    return str(s).replace("💡", "")


def quitar_sufijo(s):
    # Elimina el sufijo que agrega pandas a columnas duplicadas (".1", ".2", ...)
    return re.sub(r"\.\d+$", "", s)


def normalizar(s):
    # Colapsa cualquier espacio en blanco (incluye \xa0, \n, \t) a un solo espacio.
    return re.sub(r"\s+", " ", str(s)).strip()


def canon(s):
    return normalizar(quitar_sufijo(quitar_emoji(s)))


def extraer_nombre(celda):
    """Extrae el Nombre de una celda del roster (formato 'código\\tNombre\\tID')."""
    if celda is None:
        return None
    texto = str(celda).replace("\t", " ").strip()
    if texto == "" or texto.lower() in {"nan", "p", "d", "prueba", "asd", "-", "na", "n/a"}:
        return None
    tokens = texto.split()
    while tokens and any(ch.isdigit() for ch in tokens[0]):
        tokens.pop(0)
    while tokens and re.fullmatch(r"[\d.\-]+", tokens[-1]):
        tokens.pop()
    nombre = " ".join(tokens).strip()
    return nombre.title() if nombre else None

## Diccionario de renombrado

Usa el diccionario específico de CHM (no el `DIC_URL` de Google Sheets del
script EC). El texto real de las preguntas en el archivo 2026 difiere un poco
del diccionario en dos formas sistemáticas, que se normalizan en código:

- `"Tipo de docente implicado"` en el diccionario vs. `"Tipo de docente"` en
  el archivo real.
- Las columnas de roster (`"Tipo de docente"` e `"ID del/los docentes"`)
  terminan en un sufijo de persona (`CHM1..CHM5`, `DA1..DA10`, o `CHM`/`DA`
  para el flag binario) que no siempre coincide exactamente con los sufijos
  del diccionario (`PE1..PE6`, `DA1..DA6`). Para esas columnas se hace un
  match por "raíz" (sin sufijo) contra el diccionario y se reconstruye el
  nombre final reutilizando el sufijo real del archivo, en vez de exigir una
  coincidencia literal completa.

Columnas cuyo texto difiere del diccionario por razones de contenido (no solo
de sufijo o de la palabra "implicado") se quedan sin renombrar — no hay una
forma segura de adivinarlas.

In [ ]:
dic_path = base_dir / "notebooks" / "CHM" / "ejemplos_output" / "Momentos_EC_CP - diccionario de variables.csv"
dic = pd.read_csv(dic_path)


def canon_dic(s):
    # El diccionario dice "Tipo de docente implicado"; el archivo real 2026 dice
    # "Tipo de docente" a secas. Se normaliza acá para no tocar el CSV.
    return canon(s).replace("Tipo de docente implicado", "Tipo de docente")


_SUFFIX_RE = re.compile(r"^(.*) - (CHM\d*|DA\d*|PE\d*)$")


def _strip_suffix(s):
    m = _SUFFIX_RE.match(s)
    if m:
        return m.group(1), m.group(2)
    return s, None


# Match exacto (nombre completo, incluido el sufijo de roster si lo tiene).
dic_momentos = {canon_dic(n): nn for n, nn in zip(dic["Nombre"], dic["Nuevo Nombre"])}

# Match por raíz (sin sufijo de roster), para reconstruir el nombre final con
# el sufijo real del archivo cuando la numeración/nomenclatura no coincide
# exactamente con la del diccionario (p. ej. CHM1..CHM5/DA1..DA10 vs PE1..PE6/DA1..DA6).
dic_stem_momentos = {}
for n, nn in zip(dic["Nombre"], dic["Nuevo Nombre"]):
    n_stem, n_suf = _strip_suffix(canon_dic(n))
    if n_suf:
        nn_stem, _ = _strip_suffix(str(nn))
        dic_stem_momentos.setdefault(n_stem, nn_stem)
    else:
        # Preguntas sin sufijo en el diccionario (p. ej. "Tipo de docente") pero que
        # en el archivo real sí tienen sufijo (- CHM / - DA).
        dic_stem_momentos.setdefault(n_stem, nn)


def renombrar_columna_momento(col):
    """Busca `col` en el diccionario; si no hay match exacto, intenta por raíz
    (sin sufijo de roster) y reconstruye el nombre con el sufijo real."""
    col_c = canon_dic(col)
    if col_c in dic_momentos:
        return dic_momentos[col_c]
    stem, suf = _strip_suffix(col_c)
    if suf and stem in dic_stem_momentos:
        return f"{dic_stem_momentos[stem]} - {suf}"
    return col


print(f"diccionario: {len(dic_momentos)} entradas exactas, {len(dic_stem_momentos)} raíces")

## Listado de asistentes (roster)

El instrumento CHM es un único archivo (sin separación EC1/EC2), así que no
se agrega columna `Evento`.

In [ ]:
def construir_asistentes(df, codes):
    registros = []
    rid = df["ID de respuesta"]
    for code, rol in ROSTER_CODES.items():
        if code not in codes:
            continue
        serie = df.iloc[:, codes.index(code)]
        tmp = pd.DataFrame({"ID de respuesta": rid.values, "rol": rol, "Nombre": serie.values})
        registros.append(tmp)
    out = pd.concat(registros, ignore_index=True)
    out["Nombre"] = out["Nombre"].map(extraer_nombre)
    return out.dropna(subset=["Nombre"])


df_asistentes = construir_asistentes(df_instrumentos, codes)
print(df_asistentes.shape)
df_asistentes

## Momentos

Cada uno de los 5 bloques de momento tiene el mismo conjunto de columnas
(408). Se detectan por el ancla y se apilan en formato largo con
`Número de momento`.

In [ ]:
def construir_momentos(df):
    cols = df.columns.tolist()
    starts = [i for i, c in enumerate(cols) if canon(c) == ANCHOR]
    # Ancho del bloque: dos bloques contiguos (sin instantáneas en medio).
    width = starts[2] - starts[1]
    nombres_canon = [normalizar(quitar_emoji(c)) for c in cols[starts[0] : starts[0] + width]]

    partes = []
    for numero, s in enumerate(starts, start=1):
        blk = df.iloc[:, s : s + width].copy()
        blk.columns = nombres_canon
        blk["ID de respuesta"] = df["ID de respuesta"].values
        blk["Número de momento"] = numero
        partes.append(blk)

    out = pd.concat(partes, ignore_index=True)

    # Descarta momentos vacíos (el/la observador(a) no registró ese momento).
    col_ref = nombres_canon[0]
    out = out[out[col_ref].notna() & (out[col_ref].astype(str).str.strip() != "")]
    return out, nombres_canon


momentos, nombres_canon = construir_momentos(df_instrumentos)
print(momentos.shape)
momentos

## Reemplazar IDs (PE1..DA6) por nombres en los momentos

In [ ]:
# Columnas de "ID del/los docentes" en los momentos (valor = PE1..DA6).
id_person_cols = [c for c in nombres_canon if "ID del/los docentes" in c]

df_personas = momentos[["ID de respuesta", "Número de momento"] + id_person_cols].melt(
    id_vars=["ID de respuesta", "Número de momento"],
    var_name="persona",
    value_name="rol",
)
df_personas = df_personas.dropna(subset=["rol"])
df_personas = df_personas[df_personas["rol"].astype(str).str.strip() != ""]
df_personas["rol"] = df_personas["rol"].astype(str).str.strip()

df_personas = df_personas.merge(
    df_asistentes[["ID de respuesta", "rol", "Nombre"]],
    on=["ID de respuesta", "rol"],
    how="left",
)
df_personas = df_personas.pivot_table(
    index=["ID de respuesta", "Número de momento"],
    columns="persona",
    values="Nombre",
    aggfunc="first",
).reset_index()

momentos_nombres = momentos.drop(columns=id_person_cols).merge(
    df_personas,
    on=["ID de respuesta", "Número de momento"],
    how="left",
)
print(momentos_nombres.shape)
momentos_nombres

## Instantáneas

Cada instantánea (`I1.` ... `I18.`) tiene un bloque de columnas que se
parsean de forma semántica (no por posición). La acción es multiselección
(hasta 3), se concatena en `acción momento`.

In [ ]:
_ACTORES = {
    "Pares expertos": "Participa pares expertos",
    "Docentes acompañados": "Participa docentes acompañados",
    "Mentores": "Participa mentores",
}
_GENERO = {
    "Hombres": "Participa hombres",
    "Mujeres": "Participa mujeres",
    "Hombres y mujeres por igual": "Participa hombres y mujeres",
}
_COLS_INST = [
    "ID de respuesta",
    "Número de instantánea",
    "acción momento",
    "acción momento - otra",
    "Participa pares expertos",
    "Participa docentes acompañados",
    "Participa mentores",
    "Participa no aplica",
    "Participa hombres",
    "Participa mujeres",
    "Participa hombres y mujeres",
    "Participa género no aplica",
    "Quien dirige",
    "En qué momento",
]


def construir_instantaneas(df):
    cols = df.columns.tolist()
    grupos = {}
    for ci, name in enumerate(cols):
        m = re.match(r"^I(\d+)\.\s*(.*)$", name, flags=re.S)
        if m:
            grupos.setdefault(int(m.group(1)), []).append((ci, normalizar(m.group(2))))

    rids = df["ID de respuesta"].tolist()
    registros = []
    for ridx in range(len(df)):
        fila = df.iloc[ridx]
        for numero, items in sorted(grupos.items()):
            rec = {c: "" for c in _COLS_INST}
            rec["ID de respuesta"] = rids[ridx]
            rec["Número de instantánea"] = numero
            acciones = []
            na_count = 0
            for ci, body in items:
                val = fila.iloc[ci]
                if pd.isna(val) or str(val).strip() == "":
                    continue
                v = str(val).strip()
                if body.startswith("¿Qué acción"):
                    if body.endswith("- Otra - Texto"):
                        rec["acción momento - otra"] = v
                    else:
                        acciones.append(v)
                elif body.startswith("¿Quiénes participan"):
                    suf = body.split(" - ")[-1].strip()
                    if suf in _ACTORES:
                        rec[_ACTORES[suf]] = v
                    elif suf in _GENERO:
                        rec[_GENERO[suf]] = v
                    elif suf == "No aplica":
                        rec["Participa no aplica" if na_count == 0 else "Participa género no aplica"] = v
                        na_count += 1
                elif body.startswith("¿Se observa"):
                    rec["Quien dirige"] = v
                elif body.startswith("¿En qué momento"):
                    rec["En qué momento"] = v
            rec["acción momento"] = "; ".join(acciones)
            # Conserva la instantánea solo si tiene contenido.
            if rec["acción momento"] or rec["En qué momento"] or rec["Quien dirige"]:
                registros.append(rec)
    return pd.DataFrame(registros, columns=_COLS_INST)


instantaneas = construir_instantaneas(df_instrumentos)
print(instantaneas.shape)
instantaneas

## Metadatos del encuentro

Adaptado para CHM: `Q4` ("Nombre del Coding Hub") hace de nodo, y la
`Institución` se arma concatenando las columnas de selección múltiple
`Q5_4`...`Q5_25` (en EC era un único campo de texto libre `Q5_1`, que no
existe en este instrumento). No se construye `IDEncuentro` porque en el
script original viene de un filtro de IDs válidos por Google Sheets que
decidimos no aplicar aquí.

In [ ]:
def _col_por_regex(df, patron):
    hits = [c for c in df.columns if patron in c]
    return hits[0] if hits else None


_obs_col = _col_por_regex(df_instrumentos, "Nombre del/la observador(a)")
_nodo_idx = codes.index("Q4") if "Q4" in codes else None

meta = pd.DataFrame(
    {
        "ID de respuesta": df_instrumentos["ID de respuesta"].values,
        "Dirección IP": df_instrumentos["Dirección IP"].values,
        "Nombre del/la observador(a)": df_instrumentos[_obs_col].values if _obs_col else "",
        "Nombre del nodo": df_instrumentos.iloc[:, _nodo_idx].values if _nodo_idx is not None else "",
    }
)

# Institución: selección múltiple de sedes acompañadas (Q5_4..Q5_25), se concatena.
_inst_positions = [i for i, c in enumerate(codes) if isinstance(c, str) and c.startswith("Q5_") and c != "Q5_1"]


def _consolidar_instituciones(row):
    vals = []
    for i in _inst_positions:
        v = row.iloc[i]
        if pd.notna(v) and str(v).strip() != "":
            vals.append(str(v).strip())
    return "; ".join(vals)


meta["Institución"] = df_instrumentos.apply(_consolidar_instituciones, axis=1)

# Condiciones del espacio, fortalezas y mejoras (nivel encuentro).
for _c in df_instrumentos.columns:
    if any(_s in _c for _s in CF_STEMS):
        meta[normalizar(quitar_emoji(_c))] = df_instrumentos[_c].values

meta["nodo"] = meta["Nombre del nodo"]
meta

## Ensamblar tablas finales

In [ ]:
df_momentos = meta.drop(columns=["Dirección IP"]).merge(
    momentos_nombres, on=["ID de respuesta"], how="right"
)
# Renombra columnas de momentos y de condiciones/fortalezas/mejoras con el diccionario
# (match exacto primero, luego por raíz para columnas de roster con sufijo).
df_momentos = df_momentos.rename(columns={c: renombrar_columna_momento(c) for c in df_momentos.columns})
df_momentos = df_momentos.fillna("")
print(df_momentos.shape)
df_momentos

In [ ]:
df_instantaneas = meta[
    ["ID de respuesta", "Nombre del/la observador(a)", "nodo", "Institución"]
].merge(instantaneas, on=["ID de respuesta"], how="right")
df_instantaneas = df_instantaneas.fillna("")
print(df_instantaneas.shape)
df_instantaneas

## Guardar resultados

In [ ]:
output_dir = base_dir / "notebooks" / "CHM" / "output"
output_dir.mkdir(parents=True, exist_ok=True)

with pd.ExcelWriter(output_dir / "momentos_CHM_2026.xlsx") as writer:
    df_momentos.to_excel(writer, sheet_name="Sheet1", index=False)

df_instantaneas.to_excel(output_dir / "instantaneas_CHM_2026.xlsx", sheet_name="instantaneas_CHM_2026", index=False)
df_asistentes.to_excel(output_dir / "asistentes_CHM_2026.xlsx", sheet_name="asistentes_CHM_2026", index=False)

print("Guardado en", output_dir)
print("  momentos:", df_momentos.shape)
print("  instantaneas:", df_instantaneas.shape)
print("  asistentes:", df_asistentes.shape)